# Brasil em Dados

## Análise de padrões socioeconômicos e educacionais dos municípios brasileiros

### Pergunta central

Quais padrões socioeconômicos e educacionais existem entre os municípios
brasileiros e quais características diferenciam municípios com melhores
resultados educacionais?

### Objetivo

Integrar dados públicos do IBGE e do INEP para investigar padrões e
associações entre características socioeconômicas dos municípios e seus
indicadores educacionais.

### Observação metodológica

As análises realizadas neste trabalho buscarão identificar associações e
padrões presentes nos dados. Relações observadas entre variáveis não serão
interpretadas automaticamente como relações causais.

## 1. Configuração do ambiente

In [70]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [71]:
PROJECT_ROOT = Path("..")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"

IBGE_DIR = DATA_RAW / "ibge"
INEP_DIR = DATA_RAW / "inep"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

## 2. Dados educacionais — INEP

A base educacional utilizada contém os resultados municipais dos anos
finais do Ensino Fundamental disponibilizados pelo INEP.

A análise principal utilizará indicadores referentes ao ano de 2023 e
à rede pública, permitindo posteriormente sua integração com dados
socioeconômicos municipais do Censo Demográfico de 2022.

In [72]:
arquivo_ideb = (
    INEP_DIR
    / "divulgacao_anos_finais_municipios_2025.xlsx"
)

xls = pd.ExcelFile(arquivo_ideb)

xls.sheet_names

['IDEB_AF_MUNICÍPIOS']

In [73]:
df_ideb = pd.read_excel(
    arquivo_ideb,
    sheet_name="IDEB_AF_MUNICÍPIOS",
    header=9
)

df_ideb.head()

,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,REDE,VL_APROVACAO_2005_SI_4,VL_APROVACAO_2005_1,VL_APROVACAO_2005_2,VL_APROVACAO_2005_3,VL_APROVACAO_2005_4,VL_INDICADOR_REND_2005,...,VL_OBSERVADO_2023,VL_OBSERVADO_2025,VL_PROJECAO_2007,VL_PROJECAO_2009,VL_PROJECAO_2011,VL_PROJECAO_2013,VL_PROJECAO_2015,VL_PROJECAO_2017,VL_PROJECAO_2019,VL_PROJECAO_2021
0,RO,1100015.0,Alta Floresta D'Oeste,Estadual,75.9,69.6,75.9,83.2,77,0.76119,...,4.9,5.1,3.5,3.7,4,4.4,4.8,5,5.3,5.5
1,RO,1100015.0,Alta Floresta D'Oeste,Municipal,77.6,68.1,74.3,83.3,89,0.778453,...,4,-,3.3,3.4,3.7,4.1,4.5,4.7,5,5.3
2,RO,1100015.0,Alta Floresta D'Oeste,Pública,76.7,68.8,75,83.2,82.2,0.768449,...,4.7,5,3.5,3.7,4,4.4,4.7,5,5.3,5.5
3,RO,1100023.0,Ariquemes,Estadual,79.2,84.2,79.9,76.7,75.7,0.78989,...,4.8,4.9,3.7,3.8,4.1,4.5,4.9,5.1,5.4,5.6
4,RO,1100023.0,Ariquemes,Municipal,77.6,72.2,76.9,81.1,86.3,0.787832,...,4.4,4.5,3.3,3.5,3.7,4.1,4.5,4.8,5,5.3


## 3. Inspeção inicial da base educacional

Antes da seleção das variáveis, será examinada a estrutura da base,
incluindo dimensões, tipos das variáveis, redes disponíveis e possíveis
problemas de qualidade dos dados.

In [74]:
print(f"Linhas: {df_ideb.shape[0]}")
print(f"Colunas: {df_ideb.shape[1]}")

Linhas: 14428
Colunas: 122


In [75]:
df_ideb.info()
df_ideb["REDE"].value_counts(dropna=False)

<class 'pandas.DataFrame'>
RangeIndex: 14428 entries, 0 to 14427
Columns: 122 entries, SG_UF to VL_PROJECAO_2021
dtypes: float64(1), object(116), str(5)
memory usage: 13.4+ MB


REDE
Pública      5569
Estadual     4907
Municipal    3913
Federal        24
NaN            15
Name: count, dtype: int64

## 4. Construção da base educacional de 2023

Como a unidade de análise deste trabalho é o município, serão considerados
os indicadores referentes à rede pública.

Também será realizado um recorte temporal para o ano de 2023, mantendo
proximidade com os indicadores socioeconômicos do Censo Demográfico de 2022.

In [76]:
colunas_educacao = [
    "SG_UF",
    "CO_MUNICIPIO",
    "NO_MUNICIPIO",
    "VL_APROVACAO_2023_SI_4",
    "VL_INDICADOR_REND_2023",
    "VL_NOTA_MATEMATICA_2023",
    "VL_NOTA_PORTUGUES_2023",
    "VL_NOTA_MEDIA_2023",
    "VL_OBSERVADO_2023"
]
df_educacao = (
    df_ideb.loc[
        df_ideb["REDE"].astype("string").str.strip().eq("Pública"),
        colunas_educacao
    ]
    .copy()
)

df_educacao.head()


,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,VL_APROVACAO_2023_SI_4,VL_INDICADOR_REND_2023,VL_NOTA_MATEMATICA_2023,VL_NOTA_PORTUGUES_2023,VL_NOTA_MEDIA_2023,VL_OBSERVADO_2023
2,RO,1100015.0,Alta Floresta D'Oeste,93.2,0.931733,254.11,246.21,5.005333,4.7
5,RO,1100023.0,Ariquemes,93.8,0.937832,251.95,251.99,5.065667,4.8
7,RO,1100031.0,Cabixi,99.7,0.997227,247.93,252.78,5.011833,5
10,RO,1100049.0,Cacoal,98.8,0.988228,247.61,249.67,4.954667,4.9
13,RO,1100056.0,Cerejeiras,96.7,0.966651,255.25,252.94,5.1365,5


In [77]:
df_educacao = df_educacao.rename(
    columns={
        "SG_UF": "uf",
        "CO_MUNICIPIO": "codigo_municipio",
        "NO_MUNICIPIO": "municipio",
        "VL_APROVACAO_2023_SI_4": "taxa_aprovacao_2023",
        "VL_INDICADOR_REND_2023": "indicador_rendimento_2023",
        "VL_NOTA_MATEMATICA_2023": "nota_matematica_2023",
        "VL_NOTA_PORTUGUES_2023": "nota_portugues_2023",
        "VL_NOTA_MEDIA_2023": "nota_media_2023",
        "VL_OBSERVADO_2023": "ideb_2023"
    }
)

df_educacao.head()

,uf,codigo_municipio,municipio,taxa_aprovacao_2023,indicador_rendimento_2023,nota_matematica_2023,nota_portugues_2023,nota_media_2023,ideb_2023
2,RO,1100015.0,Alta Floresta D'Oeste,93.2,0.931733,254.11,246.21,5.005333,4.7
5,RO,1100023.0,Ariquemes,93.8,0.937832,251.95,251.99,5.065667,4.8
7,RO,1100031.0,Cabixi,99.7,0.997227,247.93,252.78,5.011833,5
10,RO,1100049.0,Cacoal,98.8,0.988228,247.61,249.67,4.954667,4.9
13,RO,1100056.0,Cerejeiras,96.7,0.966651,255.25,252.94,5.1365,5


In [78]:
colunas_numericas = [
    "taxa_aprovacao_2023",
    "indicador_rendimento_2023",
    "nota_matematica_2023",
    "nota_portugues_2023",
    "nota_media_2023",
    "ideb_2023"
]

for coluna in colunas_numericas:
    df_educacao[coluna] = pd.to_numeric(
        df_educacao[coluna],
        errors="coerce"
    )

df_educacao["codigo_municipio"] = (
    pd.to_numeric(
        df_educacao["codigo_municipio"],
        errors="coerce"
    )
    .astype("Int64")
)

df_educacao["uf"] = df_educacao["uf"].astype("string").str.strip()
df_educacao["municipio"] = df_educacao["municipio"].astype("string").str.strip()

df_educacao.dtypes

uf                            string
codigo_municipio               Int64
municipio                     string
taxa_aprovacao_2023          float64
indicador_rendimento_2023    float64
nota_matematica_2023         float64
nota_portugues_2023          float64
nota_media_2023              float64
ideb_2023                    float64
dtype: object

## 5. Qualidade dos dados educacionais

In [79]:
print(f"Número de registros: {len(df_educacao)}")
print(
    f"Número de municípios únicos: "
    f"{df_educacao['codigo_municipio'].nunique()}"
)

Número de registros: 5569
Número de municípios únicos: 5569


In [80]:
duplicados = df_educacao[
    df_educacao["codigo_municipio"].duplicated(keep=False)
]

print(f"Municípios duplicados: {len(duplicados)}")

display(duplicados.head())

Municípios duplicados: 0


,uf,codigo_municipio,municipio,taxa_aprovacao_2023,indicador_rendimento_2023,nota_matematica_2023,nota_portugues_2023,nota_media_2023,ideb_2023


In [81]:
ausentes = (
    df_educacao
    .isna()
    .sum()
    .to_frame("n_ausentes")
)

ausentes["percentual"] = (
    100 * ausentes["n_ausentes"] / len(df_educacao)
)

ausentes = ausentes.sort_values(
    "percentual",
    ascending=False
)

display(ausentes)

,n_ausentes,percentual
ideb_2023,187,3.357874
nota_matematica_2023,186,3.339917
nota_media_2023,186,3.339917
nota_portugues_2023,186,3.339917
taxa_aprovacao_2023,49,0.879871
indicador_rendimento_2023,49,0.879871
uf,0,0.000000
codigo_municipio,0,0.000000
municipio,0,0.000000


In [82]:
display(
    df_educacao[colunas_numericas]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
taxa_aprovacao_2023,5520.0,93.214348,6.686405,57.200000,89.900000,95.200000,98.600000,100.000000
indicador_rendimento_2023,5520.0,0.931812,0.067018,0.566969,0.897912,0.951581,0.985695,1.000000
nota_matematica_2023,5383.0,252.515215,19.103108,190.200000,239.040000,251.980000,264.390000,404.890000
nota_portugues_2023,5383.0,253.072422,16.488468,196.320000,241.855000,253.620000,264.415000,362.020000
nota_media_2023,5383.0,5.093112,0.580840,3.116167,4.696416,5.100333,5.471750,9.299333
ideb_2023,5382.0,4.765831,0.734195,2.600000,4.300000,4.800000,5.300000,9.300000


In [83]:
print(
    "Taxas de aprovação fora de 0–100:",
    (
        (df_educacao["taxa_aprovacao_2023"] < 0)
        |
        (df_educacao["taxa_aprovacao_2023"] > 100)
    ).sum()
)

print(
    "Valores de IDEB negativos:",
    (df_educacao["ideb_2023"] < 0).sum()
)

Taxas de aprovação fora de 0–100: 0
Valores de IDEB negativos: 0


In [84]:
print(
    "Taxas de aprovação fora de 0–100:",
    (
        (df_educacao["taxa_aprovacao_2023"] < 0)
        |
        (df_educacao["taxa_aprovacao_2023"] > 100)
    ).sum()
)

print(
    "Valores de IDEB negativos:",
    (df_educacao["ideb_2023"] < 0).sum()
)

Taxas de aprovação fora de 0–100: 0
Valores de IDEB negativos: 0


In [85]:
df_verificacao = df_educacao[
    [
        "nota_media_2023",
        "indicador_rendimento_2023",
        "ideb_2023"
    ]
].dropna().copy()

df_verificacao["ideb_calculado"] = (
    df_verificacao["nota_media_2023"]
    * df_verificacao["indicador_rendimento_2023"]
)

df_verificacao["erro"] = (
    df_verificacao["ideb_2023"]
    - df_verificacao["ideb_calculado"]
).abs()

df_verificacao["erro"].describe()

count    5382.000000
mean        0.025214
std         0.014442
min         0.000000
25%         0.012759
50%         0.025569
75%         0.037790
max         0.049992
Name: erro, dtype: float64

In [86]:
arquivo_educacao_processado = (
    DATA_PROCESSED / "educacao_municipios_2023.csv"
)

df_educacao.to_csv(
    arquivo_educacao_processado,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Base educacional salva em:",
    arquivo_educacao_processado.resolve()
)

Base educacional salva em: C:\Users\manel\Documents\Ciencia de Dados\brasil_em_dados\data\processed\educacao_municipios_2023.csv


## 6. Dados socioeconômicos — IBGE

Para caracterizar o contexto socioeconômico dos municípios, serão
utilizados dados do Censo Demográfico de 2022 disponibilizados pelo IBGE.

Inicialmente serão consideradas duas características:

- população residente;
- rendimento domiciliar mensal per capita.

A escolha de dados referentes a 2022 mantém proximidade temporal com os
indicadores educacionais de 2023 utilizados na análise.

## 7. Integração das bases INEP e IBGE

Após o tratamento independente das bases educacional e socioeconômica,
os dados serão integrados utilizando o código do município do IBGE como
chave.

Antes da análise, será verificado se existem municípios presentes em
apenas uma das fontes, evitando a exclusão silenciosa de registros.